# Overview

This notebook id dedicated for evaluating NER model on a benchmark dataset.

# Step 0 - Setup
Run the code below to mount your Google Drive and most of the necessary packages to carry out the evaluation

**Action:**
No code changes required. When prompted, connect your Google account

In [4]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets


In [5]:
%%capture
!cd /content/
!rm -rf ./CASM_utils/
!pip install git+https://github.com/ay94/multilingual-ner.git!pip install -e CASM_utils/
!cd /content/CASM_utils


import CASM_utils
import importlib
from CASM_utils import utils, ner
importlib.reload(ner)

In [6]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import os
import pandas as pd

Mounted at /content/drive/


In [7]:
# direct the file handler to the data folder
FOLDER = '/content/drive/MyDrive/CASM/FAST/Thai/NER/Benchmark'
fh = utils.FileHandler(FOLDER)

 # Read NER Dataset --> the output should be list of words corresponding to list of labels
 Read NER data class, it contains three different functionalities to read NER data.
  The NER data in the literature normally have consistent internal structure and flexible external structure.
  The internal structure is that it comes in word-label pair, this is consistent across all datasets.
  The external structure normally differ from dataset to another, which is divided to three main categories:
  - Data that comes in one text file, the read_ner_file function can be used in this case.
  - Data that comes in text files split into, train, val and test, this type you can either read individual file separately or put them all in one folder and read_ner_directory function.
  - Data that comes in directory where the directory contians various text files divide by topic (e.g, AQMAR), this type of data normally wikipedia articles that has been scraped and preprocessed into named entities structure.
  - Finally data available on huggingface and this can be loaded using load_dataset function and pass it to the read_dataset class method.
  
  Most of the datasets fall under one of these types if your data is different you can add function to this class dedicated to your data.

pythainlp/thainer-corpus-v2

In [8]:
thainer_label_map = {
    'B-PERSON': 0, 'I-PERSON': 1, 'O': 2,
    'B-ORGANIZATION': 3, 'I-ORGANIZATION': 5,
    'B-LOCATION': 4, 'I-LOCATION': 6,
    'B-DATE': 7, 'I-DATE': 8,
    'B-TIME': 9, 'I-TIME': 10,
    'B-MONEY': 11, 'I-MONEY': 12,
    'B-FACILITY': 13, 'I-FACILITY': 14,
    'B-URL': 15, 'I-URL': 16,
    'B-PERCENT': 17, 'I-PERCENT': 18,
    'B-LEN': 19, 'I-LEN': 20,
    'B-AGO': 21, 'I-AGO': 22,
    'B-LAW': 23, 'I-LAW': 24,
    'B-PHONE': 25, 'I-PHONE': 26,
    'B-EMAIL': 27, 'I-EMAIL': 28,
    'B-ZIP': 29, 'B-TEMPERATURE': 30, 'I-TEMPERATURE': 31,
    'B-DTAE': 32, 'I-DTAE': 33,
    'B-DATA': 34
}

thainer = ner.ReadNERData()
thainer_words, thainer_labels = thainer.read_dataset('pythainlp/thainer-corpus-v2', thainer_label_map)

Generating train split:   0%|          | 0/3938 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1313 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1313 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/1313 [00:00<?, ?it/s]

### Check for dataset alignment
The first thing to do after loading the data is to check that it is aligned with the standard annotation scheme using check_labels function. NER datasets have various annotation schemes, the standard one we are interested in is the conll annotation scheme where the data should be divided into, *LOC*, *PERS*, *ORG*, *MISC* entities and each entity have BI boudary (e.g, B-LOC, I-LOC) and outside named entity O. This is the standard annotation scheme we are aiming for and some dataset comes with fine grained annotations or even different labels. This requires realigning the dataset labels to the standard scheme by defining a dataset label alignment dictionary and use the align_dataset function.

pythainlp/thainer-corpus-v2

In [9]:
print(ner.check_labels(thainer_labels))

thainer_label_map = {
    'B-PERSON': 'B-PER', 'I-PERSON': 'I-PER',
    'B-ORGANIZATION': 'B-ORG', 'I-ORGANIZATION': 'I-ORG',
    'B-LOCATION': 'B-LOC', 'I-LOCATION': 'I-LOC',
    'B-DATE': 'O', 'I-DATE': 'O',
    'B-TIME': 'O', 'I-TIME': 'O',
    'B-MONEY': 'O', 'I-MONEY': 'O',
    'B-FACILITY': 'O', 'I-FACILITY': 'O',
    'B-URL': 'O', 'I-URL': 'O',
    'B-PERCENT': 'O', 'I-PERCENT': 'O',
    'B-LEN': 'O', 'I-LEN': 'O',
    'B-AGO': 'O', 'I-AGO': 'O',
    'B-LAW': 'O', 'I-LAW': 'O',
    'B-PHONE': 'O', 'I-PHONE': 'O',
    'B-EMAIL': 'O', 'I-EMAIL': 'O',
    'B-ZIP': 'O', 'B-TEMPERATURE': 'O', 'I-TEMPERATURE': 'O',
    'B-DTAE': 'O', 'I-DTAE': 'O',
    'B-DATA': 'O', 'O': 'O'
}


# Align the dataset labels to the standard labels
thainer_labels = ner.align_dataset(thainer_labels, thainer_label_map)
print(ner.check_labels(thainer_labels))


{'I-ORGANIZATION', 'I-TEMPERATURE', 'B-FACILITY', 'I-LAW', 'I-EMAIL', 'B-URL', 'B-LAW', 'I-AGO', 'B-LEN', 'B-DTAE', 'I-LEN', 'I-MONEY', 'B-EMAIL', 'O', 'B-ZIP', 'B-PERSON', 'B-AGO', 'I-DATE', 'I-TIME', 'I-FACILITY', 'B-PERCENT', 'I-URL', 'I-PHONE', 'B-DATE', 'B-TEMPERATURE', 'B-TIME', 'B-LOCATION', 'I-PERSON', 'I-LOCATION', 'I-PERCENT', 'B-MONEY', 'B-PHONE', 'B-ORGANIZATION'}
{'I-ORG', 'B-PER', 'O', 'I-LOC', 'B-ORG', 'I-PER', 'B-LOC'}


# Model Evaluation
Model evaluation is dvided into three steps:
- Loading the model using get_model funtion
- Generating the evaluation benchmark using generate_evaluation_data function
- Apply the model to the benchmakr and compute the performance using eval_fn

All of these steps can be achieved by calling evaluate_model function. It is worth noting that all models comes with their own labeling scheme and some models have different annotation scheme from the standard one we are using, this requires using label alignment dictionary to align the model's output.

In [10]:
model_name = "Pavarissy/phayathaibert-thainer"
model_name_output = 'Pavarissy-phayathaibert-thainer'
model_evaluation = ner.ModelEvaluation(model_name, thainer_label_map)

tokenizer_config.json:   0%|          | 0.00/144k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.18k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

In [11]:
model_evaluation.model.config.id2label

{0: 'B-PERSON',
 1: 'I-PERSON',
 2: 'O',
 3: 'B-ORGANIZATION',
 4: 'B-LOCATION',
 5: 'I-ORGANIZATION',
 6: 'I-LOCATION',
 7: 'B-DATE',
 8: 'I-DATE',
 9: 'B-TIME',
 10: 'I-TIME',
 11: 'B-MONEY',
 12: 'I-MONEY',
 13: 'B-FACILITY',
 14: 'I-FACILITY',
 15: 'B-URL',
 16: 'I-URL',
 17: 'B-PERCENT',
 18: 'I-PERCENT',
 19: 'B-LEN',
 20: 'I-LEN',
 21: 'B-AGO',
 22: 'I-AGO',
 23: 'B-LAW',
 24: 'I-LAW',
 25: 'B-PHONE',
 26: 'I-PHONE',
 27: 'B-EMAIL',
 28: 'I-EMAIL',
 29: 'B-ZIP',
 30: 'B-TEMPERATURE',
 31: 'I-TEMPERATURE',
 32: 'B-DTAE',
 33: 'I-DTAE',
 34: 'B-DATA',
 35: 'I-DATA'}

In [12]:
fh.create_folder(f'outputs/{model_name_output}')

Folder 'outputs/Pavarissy-phayathaibert-thainer' created successfully.


#### pythainlp/thainer-corpus-v2

In [13]:
data_name = "thainer"
thainer_evaluation_output = model_evaluation.evaluate_model(thainer_words, thainer_labels)

  0%|          | 0/83 [00:00<?, ?it/s]

In [14]:
thainer_seqeval = thainer_evaluation_output.get_classification('Seqeval')
thainer_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.8528,0.8975,0.8746,839
1,ORG,0.8753,0.9175,0.8959,1285
2,PER,0.9138,0.9364,0.9250,645
3,micro,0.8772,0.9159,0.8961,2769
4,macro,0.8806,0.9171,0.8985,2769
5,weighted,0.8774,0.9159,0.8962,2769


In [15]:
thainer_sklearn = thainer_evaluation_output.get_classification('Sklearn')
thainer_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8764,0.9046,0.8903,839
1,B-ORG,0.9095,0.9385,0.9238,1285
2,B-PER,0.9588,0.9752,0.9669,645
3,I-LOC,0.8587,0.8256,0.8418,780
4,I-ORG,0.8540,0.9154,0.8836,1265
5,I-PER,0.9830,0.9914,0.9872,2448
6,O,0.9939,0.9903,0.9921,44252
7,accuracy,0.9831,51514,None,None
8,macro,0.9192,0.9344,0.9265,51514
9,weighted,0.9834,0.9831,0.9832,51514


In [16]:
thainer_seqeval.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-seqeval.csv'),
    index=False
)
thainer_sklearn.to_csv(
    fh.cr_fn(f'outputs/{model_name_output}/{data_name}-sklearn.csv'),
    index=False
)